# Detección de objetos usando redes preentrenadas &#128248;

En este notebook se va a hacer un detector de objetos en imágenes utilizando una red preentrenada.

Esta técnica utiliza la red MobileNet SSD, que es un modelo ligero de detección de objetos en tiempo real que combina la arquitectura eficiente MobileNet con el detector Single-Shot Multibox Detector (SSD). Diseñado para dispositivos móviles y embebidos, utiliza convoluciones separables en profundidad para equilibrar velocidad y precisión, siendo capaz de detectar 20 clases comunes de objetos (como personas, vehículos, animales) en una sola pasada. 

## Características de la red MobileNet SSD.
- **Eficiencia**: Diseñado para alta velocidad de fotogramas (FPS) con bajo consumo de memoria.
- **Arquitectura**: Usa MobileNet como red base (backbone) y SSD para localizar y clasificar objetos.
- **Entrada y Salida**: Generalmente toma imágenes de píxeles.
- **Implementación**: Comúnmente implementado usando Caffe o TensorFlow. 

## Proyecto 1: detección objetos en una fotografía.

Requisitos mínimos:   

pip install opencv-python

In [ ]:
!pip install opencv-python

### Importamos librerías.

In [1]:
import cv2

### Cargamos red DNN MobileNet SSD

In [2]:
# Arquitectura
prototxt = "./modelos/MobileNetSSD_deploy.prototxt.txt"
# Pesos preentrenados
model = "./modelos/MobileNetSSD_deploy.caffemodel"
# Etiquetas de las clases
classes = {0:"background", 1:"aeroplane", 2:"bicycle",
          3:"bird", 4:"boat",
          5:"bottle", 6:"bus",
          7:"car", 8:"cat",
          9:"chair", 10:"cow",
          11:"diningtable", 12:"dog",
          13:"horse", 14:"motorbike",
          15:"person", 16:"pottedplant",
          17:"sheep", 18:"sofa",
          19:"train", 20:"tvmonitor"}
# Cargar la red
net = cv2.dnn.readNetFromCaffe(prototxt, model)


### Cargar imagen y realizar preprocesado

#### BLOB-Concepto   
Un `blob` (*Binary Large OBject*) en `OpenCV DNN` es básicamente una imagen preprocesada y formateada para ser consumida por una red neuronal.   

En el siguiente código,esto es lo que hace `blobFromImage`
Toma la imagen y aplica varias transformaciones en un solo paso:

1. Reescala los valores de píxel — el parámetro 0.007843 (≈ 1/127.5) multiplica cada píxel para normalizar los valores
2. Resta la media — (127.5, 127.5, 127.5) se resta a cada canal (BGR), centrando los valores alrededor de 0
3. Cambia el orden de dimensiones — de (H, W, C) a (N, C, H, W):

    - N = número de imágenes (batch)
    - C = canales de color
    - H = altura
    - W = ancho

Así pues, la imagen original con formato *Height*, *Widht*, *Channels*(canales de color) - (300, 300, 3) se convierte en otra de formato  (1, 3, 300, 300).   

De esta forma se *adapta* el formato a lo que espera la red neuronal (formato 'típico' NCHW) con valores normalizados. Es el modo de que el modelo funcione correctamente, dado que el entrenamiento previo se ha realizado en dicho formato.   

En definitiva, se traduce el formato de la imagen original al formato que 'entiende' la red neuronal.

In [ ]:
image = cv2.imread("./img/imagentest.jpg")
height, width, _ = image.shape
image_resized = cv2.resize(image, (300, 300))
# Create a blob
blob = cv2.dnn.blobFromImage(image_resized, 0.007843, (300, 300), (127.5, 127.5, 127.5)) # Normalizamos la imagen y la convertimos a un blob
print("blob.shape:", blob.shape)

### Detección y predicción

Seguidamente se pasa el blob ya prepocesado a la RN y se ejecuta la inferecia (`detections=net.forward()`), devolviendo las detecciones que la red encuentra.   

La variable `detections`tiene una forma (1,1,N,7), donde N es el número posible de detecciones y 7 son los datos de cada una de estas.   

El bucle que sigue realiza la iteración sobre cada detección, teniendo en cuenta que `detections[0][0]`elimina las dos primeras dimensiones irrelevantes (batch y placeholder), con lo que se queda un array de (N,7) para cada elemento de `detection`, con la siguiente información:
- [0] --> ID de la imagen dentro del batch.
- [1] --> Clase detectada (número).
- [2] --> Confianza (valor entre 0 y 1).
- [3:7] --> Coordenadas del bounding box (normalizadas entre 0-1)

Filtrado por nivel de confianza, en este caso sólo se procesan las detecciones con un nivel de confianza superior al 45%.
(`if detection[2]>0.45`) intentado evitar posibles *falsos positivos*

Obtención de la etiqueta de la clase. (`label = classes[detection[1]]`); `detection[1]` es un número (ej: 15). Se usa como índice en la lista clases para obtener el nombre legible (ej: "person").

Se calculan las coordenadas del `bounding box`. Al multiplicarlas por `[width, height, width, height]` se convierten a píxeles reales de la imagen original.Las coordenadas que devuelve la red están **normalizadas** entre 0 y 1 (relativas al tamaño de imagen).     

Despues se convierten a enteros.     

Finalmente, se dibuja la `caja` de color verde(255, 0, 0) sobre la imagen usando las esquinas que se han calculado. El 2 es el grosor del 'borde'. Se escribe la confianza y la etiqueta sobre la imagen, posicionándolo sobre el borde superior de la bounding box.    

Las tres últimas líneas nos muestran la imagen, quedando la aplicación a la espera de ser cerrada.

In [ ]:
net.setInput(blob)
detections = net.forward()
for detection in detections[0][0]:
     print(detection)
     if detection[2] > 0.45: # Filtramos las detecciones con una confianza mayor a 0.45
          label = classes[detection[1]]
          print("Label:", label)
          box = detection[3:7] * [width, height, width, height] # creamos las coordenadas del bounding box
          x_start, y_start, x_end, y_end = int(box[0]), int(box[1]), int(box[2]), int(box[3]) # convertimos las coordenadas a enteros
          cv2.rectangle(image, (x_start, y_start), (x_end, y_end), (0, 255, 0), 2)
          cv2.putText(image, "Conf: {:.2f}".format(detection[2] * 100), (x_start, y_start - 5), 1, 1.2, (255, 0, 0), 2)
          cv2.putText(image, label, (x_start, y_start - 25), 1, 1.2, (255, 0, 0), 2)
cv2.imshow("Image", image)
cv2.waitKey(0)
cv2.destroyAllWindows()

## Proyecto 2: detección objetos en un vídeo.

### Cargar el video y preprocesado

[Enlace a videos de prueba](https://github.com/intel-iot-devkit/sample-videos/tree/master)

En este proyecto, lo que se analiza es un video, procediendo a la detección también de objetos.

Lo primero a realizar es capturar el video  usando un objeto cap sobre el fichero mp4, de forma que se pueda leer el vídeo fotograma a fotograma.   
El bucle `while` corre indefinidamente.En cada iteración, `cap.read()` lee el siguiente fotograma del video y devuelve dos cosas:

- **ret**: booleano — True si el fotograma se leyó correctamente, False si el video terminó o hubo un error
- **frame**: la imagen del fotograma actual como array numpy.   

La linea `if ret == False: break`nos proporciona la condición de parada al finalizar el video.

Se preprocesa el fotograma, de forma similar a lo realizado con la imagen estática, pero ahora realizado sobre cada fotograma. La secuencia es la siguiente:   
- Se guarda el tamaño original para luego escalar el bounding box
- Se redimensiona a 300×300 para que la red lo acepte
- Se convierte a blob normalizando los valores de píxel  

Lo siguiente es realizar la **detección** y la **predicción**, pasando el blob a la RN y ejecutando la inferencia. Nuevamente, se ejecuta para cada fotograma, por lo que la red realiza la detección de objetos de forma continuada y en ***tiempo real***.

Se dibuja la `bounding box` y se pone la etiqueta y la confianza, pero ahora se dibuja sobre cada frame en lugar de sobre una imagen fija.   

Finalmente, se muestra el fotograma y se *escucha* la tecla `ESC` por si el usuario desea salir manualmente del bucle. Para ello espera un milisegundo (`waitKey(1)`), permitiendo que el video *fluya* generando el efecto de video continuo.   

Como última medida, se liberan recursos con `cap.release()` y `cv2.destroyAllWindows()`

In [ ]:
cap = cv2.VideoCapture("./video/car-detection.mp4")
while True:
     ret, frame, = cap.read()
     if ret == False:
          break
     # Preprocesamiento del frame
     height, width, _ = frame.shape
     frame_resized = cv2.resize(frame, (300, 300))
     # Creación del blob y normalización del frame
     blob = cv2.dnn.blobFromImage(frame_resized, 0.007843, (300, 300), (127.5, 127.5, 127.5))
     #print("blob.shape:", blob.shape)
     # ----------- DETECCION Y PREDICCION -----------
     net.setInput(blob) 
     detections = net.forward()
     for detection in detections[0][0]:
          #print(detection)
          if detection[2] > 0.45:
               label = classes[detection[1]]
               #print("Label:", label)
               box = detection[3:7] * [width, height, width, height]
               x_start, y_start, x_end, y_end = int(box[0]), int(box[1]), int(box[2]), int(box[3])
               cv2.rectangle(frame, (x_start, y_start), (x_end, y_end), (0, 255, 0), 2)
               cv2.putText(frame, "Conf: {:.2f}".format(detection[2] * 100), (x_start, y_start - 5), 1, 1.2, (255, 0, 0), 2)
               cv2.putText(frame, label, (x_start, y_start - 25), 1, 1.5, (0, 255, 255), 2)
     cv2.imshow("Frame", frame)
     if cv2.waitKey(1) & 0xFF == 27: # Detectamos la tecla ESC (0xFF=27, código ASCII 27) para salir del bucle
          break
cap.release()
cv2.destroyAllWindows()